In [ ]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path().resolve().parent

ruta_dataset = BASE_DIR / "data" / "merged_features_final.csv"

df = pd.read_csv(ruta_dataset, sep=";")
df

### Análisis del Dataset

A continuación análisamos los datos del dataset para determinar manualmente buenos parámetros para puntuar url de phshing.

#### Entropía

In [ ]:
mean_entropy_leg = df[df.label == 'legitimate'].entropy.mean()
mean_entropy_phi = df[df.label == 'phishing'].entropy.mean()
max_entropy_leg = df[df.label == 'legitimate'].entropy.max()


print("Promedio de entropía para url legítimas:")
print(mean_entropy_leg)
print("Promedio de entropía para url phishing")
print(mean_entropy_phi)
print("\n")

print("Resumen entropía url legítimas")
print(df[df.label == 'legitimate'].entropy.describe())

print("\nResumen entropía url phishing")
print(df[df.label == 'phishing'].entropy.describe())

#### Largo URL's

In [ ]:
print("Resumen largo url legítimas")
print(df[df.label == 'legitimate'].url_length.describe())

print("\nResumen largo url phishing")
print(df[df.label == 'phishing'].url_length.describe())

#### Subdominios

In [ ]:
print("Resumen número de subdominios legítimas")
print(df[df.label == 'legitimate'].num_subdomains.describe())

print("\nResumen número de subdominios phishing")
print(df[df.label == 'phishing'].num_subdomains.describe())

#### Largo host

In [ ]:
print("Resumen hostname length legítimas")
print(df[df.label == 'legitimate'].length_hostname.describe())

print("\nResumen hostname length phishing")
print(df[df.label == 'phishing'].length_hostname.describe())

#### Digit ratio

In [ ]:
print("Resumen proporción de dígitos")
print(df[df.label == 'legitimate'].ratio_digits_url.describe())

print("\nResumen proporción de dígitos")
print(df[df.label == 'phishing'].ratio_digits_url.describe())

### Análisis de resultados del test

In [ ]:

BASE_DIR = Path().resolve().parent

ruta_results = BASE_DIR / "outputs" / "static_phishing_test.csv"

df_test = pd.read_csv(ruta_results)
df_test

In [14]:
print(df_test[df_test.label == 'legitimate'].static_phishing_score.describe())
print("\n")
print(df_test[df_test.label == 'phishing'].static_phishing_score.describe())

count    28359.000000
mean         0.082369
std          0.029070
min          0.062000
25%          0.062000
50%          0.074000
75%          0.092000
max          0.369000
Name: static_phishing_score, dtype: float64


count    31641.000000
mean         0.212363
std          0.119818
min          0.062000
25%          0.110000
50%          0.162000
75%          0.310000
max          0.675000
Name: static_phishing_score, dtype: float64


Si ponemos el umbral en $0.10$ para determinar phishing, tenemos

In [15]:
umbral = 0.10

legitimate_estimation = df_test[df_test.static_phishing_score <= umbral]
phishing_estimation = df_test[df_test.static_phishing_score > umbral]

TP = len(phishing_estimation[phishing_estimation.label == 'phishing']) #true positive
FP = len(phishing_estimation[phishing_estimation.label == 'legitimate']) # false positive
FN = len(legitimate_estimation[legitimate_estimation.label == 'phishing']) # false negative
TN = len(legitimate_estimation[legitimate_estimation.label == 'legitimate']) # true positive

precision = TP / (TP + FP)
recall = TP / (TP + FN)
FPR = FP / (FP + TN)

# La precisión es la tasa de url clasificadas como phishing que realmente lo son
print("Precision", round(precision, 4) * 100, "%")

# El recall es la tasa de casos de phishing que se detectaron
print("Recall", round(recall, 4) * 100, "%")

# FPR es la tasa de falsos positivos
print("False Positive Rate", round(FPR, 4) * 100, "%")

Precision 90.18 %
Recall 85.95 %
False Positive Rate 10.45 %


In [ ]:
import matplotlib.pyplot as plt

umbrales = [0.08, 0.09, 0.10, 0.11, 0.12, 0.14, 0.16, 0.18, 0.20]

fprs = []
recalls = []

for umbral in umbrales:
    legitimate_estimation = df_test[df_test.static_phishing_score <= umbral]
    phishing_estimation = df_test[df_test.static_phishing_score > umbral]

    TP = len(phishing_estimation[phishing_estimation.label == 'phishing'])
    FP = len(phishing_estimation[phishing_estimation.label == 'legitimate'])
    FN = len(legitimate_estimation[legitimate_estimation.label == 'phishing'])
    TN = len(legitimate_estimation[legitimate_estimation.label == 'legitimate'])

    precision = TP / (TP + FP) if TP + FP else 0
    recall = TP / (TP + FN) if TP + FN else 0
    FPR = FP / (FP + TN) if FP + TN else 0

    fprs.append(FPR)
    recalls.append(recall)

    print(f"Umbral: {umbral:.2f} | Precision: {precision:.4f} | Recall: {recall:.4f} | FPR: {FPR:.4f}")

plt.plot(fprs, recalls, marker='o')
plt.xlabel("False Positive Rate")
plt.ylabel("Recall")
plt.title("Recall vs FPR para distintos umbrales")
plt.grid(True)
plt.show()

#El que lo minimiza resulta ser el umbral 0.10